# 🔭 Module 8 — Lab: AgentOps & Production Observability

**Mục tiêu:** Thêm full observability stack vào LoanBot v3.0:
- ✅ **Trace Collector** — thu thập spans từ 8 agents, propagate trace_id
- ✅ **AgentMetricsCollector** — latency, tokens, cost, error rate per agent
- ✅ **SLO Monitor** — track error budget, alert khi SLO bị vi phạm
- ✅ **Cost Tracker** — real-time cost per hồ sơ từ Claude API
- ✅ **Dashboard Renderer** — in observability summary ra console (mô phỏng Grafana)

---
```
Thời gian ước tính: 40-50 phút
API: Claude claude-haiku-4-5-20251001
Packages: anthropic, time, statistics, dataclasses (built-in)
```

## ⚙️ Setup

In [ ]:
!pip install anthropic --quiet

import os, json, time, uuid, statistics, asyncio
import anthropic
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Any
from collections import defaultdict
from enum import Enum

os.environ["ANTHROPIC_API_KEY"] = "sk-ant-api03-YOUR_KEY_HERE"
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"

print("✅ Setup hoàn tất!")
print(f"📅 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 🔷 Phần 1 — Span & Trace Collector

Mô phỏng OpenTelemetry: tạo Spans, propagate trace_id qua 8 agents, hiển thị waterfall.

In [ ]:
# ─── Span — đơn vị cơ bản của trace

@dataclass
class Span:
    span_id: str
    trace_id: str
    parent_id: Optional[str]
    name: str
    start_time: float = field(default_factory=time.time)
    end_time: Optional[float] = None
    attributes: Dict[str, Any] = field(default_factory=dict)
    status: str = "OK"   # OK | ERROR

    def end(self, status: str = "OK", **attrs):
        self.end_time = time.time()
        self.status = status
        self.attributes.update(attrs)

    @property
    def duration_ms(self) -> float:
        if self.end_time is None:
            return (time.time() - self.start_time) * 1000
        return (self.end_time - self.start_time) * 1000


class TraceCollector:
    """In-memory trace store (production → export to Jaeger/Phoenix)."""

    def __init__(self):
        self._spans: List[Span] = []

    def start_span(self, name: str, trace_id: str, parent_id: Optional[str] = None, **attrs) -> Span:
        span = Span(
            span_id=str(uuid.uuid4())[:8],
            trace_id=trace_id,
            parent_id=parent_id,
            name=name,
            attributes=attrs
        )
        self._spans.append(span)
        return span

    def get_trace(self, trace_id: str) -> List[Span]:
        return [s for s in self._spans if s.trace_id == trace_id]

    def print_waterfall(self, trace_id: str):
        spans = sorted(self.get_trace(trace_id), key=lambda s: s.start_time)
        if not spans:
            print("No spans found")
            return
        t0 = spans[0].start_time
        total_ms = (spans[-1].end_time or time.time() - t0) * 1000
        print(f"\n🔭 Trace Waterfall — {trace_id}")
        print(f"{'Span Name':<24} {'Start':>8} {'Duration':>10} Status")
        print("─" * 60)
        for s in spans:
            indent = "  " if s.parent_id else ""
            start_rel = (s.start_time - t0) * 1000
            dur = s.duration_ms
            status_icon = "✅" if s.status == "OK" else "❌"
            tokens = s.attributes.get('llm.tokens.total', '-')
            cost = s.attributes.get('llm.cost.usd', '')
            cost_str = f"  ${cost:.5f}" if cost else ""
            print(f"{indent}{s.name:<22} {start_rel:>8.0f}ms {dur:>8.0f}ms {status_icon} tok:{tokens}{cost_str}")
        print(f"{'Total Pipeline':<24} {'0':>8} {(spans[-1].end_time - t0)*1000 if spans[-1].end_time else 0:>8.0f}ms")


# Khởi tạo global collector
tracer = TraceCollector()
print("✅ TraceCollector ready")

---
## 🔷 Phần 2 — AgentMetricsCollector (4 Golden Signals)

In [ ]:
# ─── AgentMetricsCollector — track 4 Golden Signals

class AgentMetricsCollector:
    """Production → export to Prometheus. Lab → in-memory."""

    # LoanBot v3.0 SLOs
    SLO_LATENCY_MS = {
        "CreditAgent": 300, "IncomeAgent": 400, "FraudAgent": 250,
        "RiskAgent": 500, "ComplianceAgent": 200, "ReportAgent": 800,
        "AuditAgent": 150, "IntelligentSupervisor": 5000
    }
    SLO_ERROR_RATE = 0.1   # 0.1%
    SLO_COST_PER_APP = 0.01  # $0.01/hồ sơ

    def __init__(self):
        self.latencies  = defaultdict(list)
        self.errors     = defaultdict(int)
        self.tokens_in  = defaultdict(int)
        self.tokens_out = defaultdict(int)
        self.costs      = defaultdict(float)
        self.calls      = defaultdict(int)
        self.total_apps = 0
        self.total_cost = 0.0

    def record(self, agent_name: str, latency_ms: float, tokens_in: int = 0,
               tokens_out: int = 0, cost_usd: float = 0.0, error: bool = False):
        self.latencies[agent_name].append(latency_ms)
        self.tokens_in[agent_name]  += tokens_in
        self.tokens_out[agent_name] += tokens_out
        self.costs[agent_name]      += cost_usd
        self.calls[agent_name]      += 1
        self.total_cost             += cost_usd
        if error:
            self.errors[agent_name] += 1

    def record_app_complete(self):
        self.total_apps += 1

    def p(self, agent_name: str, percentile: float = 99) -> float:
        lats = sorted(self.latencies[agent_name])
        if not lats: return 0.0
        idx = max(0, int(percentile / 100 * len(lats)) - 1)
        return lats[min(idx, len(lats) - 1)]

    def error_rate(self, agent_name: str) -> float:
        total = self.calls[agent_name]
        return (self.errors[agent_name] / total * 100) if total else 0.0

    def slo_status(self, agent_name: str) -> str:
        p99 = self.p(agent_name, 99)
        slo = self.SLO_LATENCY_MS.get(agent_name, 1000)
        if p99 == 0: return "N/A"
        ratio = p99 / slo
        if ratio < 0.9: return "✅ OK"
        if ratio < 1.0: return "⚠️ WARN"
        return "❌ BREACH"

    def cost_per_app(self) -> float:
        return self.total_cost / max(self.total_apps, 1)

    def print_dashboard(self):
        print("\n" + "="*80)
        print("📊 LOANBOT v3.0 — AGENTOPS DASHBOARD")
        print("="*80)

        # Top stats
        cpa = self.cost_per_app()
        cpa_status = "✅" if cpa < self.SLO_COST_PER_APP else "❌"
        print(f"\n💰 Cost/Hồ Sơ: ${cpa:.5f} {cpa_status} | Total Apps: {self.total_apps} | Total Cost: ${self.total_cost:.4f}")

        # Per-agent table
        print(f"\n{'Agent':<22} {'Calls':>5} {'P50(ms)':>8} {'P99(ms)':>8} {'Err%':>6} {'Tokens':>8} {'Cost($)':>9} {'SLO':>10}")
        print("─" * 82)
        for name in sorted(self.calls.keys()):
            tok_total = self.tokens_in[name] + self.tokens_out[name]
            print(f"{name:<22} {self.calls[name]:>5} {self.p(name,50):>8.0f} {self.p(name,99):>8.0f}"
                  f" {self.error_rate(name):>5.2f}% {tok_total:>8} {self.costs[name]:>9.5f} {self.slo_status(name):>10}")

        # Error budget
        total_calls = sum(self.calls.values())
        total_errors = sum(self.errors.values())
        overall_er = (total_errors / total_calls * 100) if total_calls else 0
        budget_used = overall_er / self.SLO_ERROR_RATE * 100
        print(f"\n🎯 Error Budget: {max(0, 100-budget_used):.1f}% remaining | Overall error rate: {overall_er:.3f}%")
        print("="*80)


metrics = AgentMetricsCollector()
print("✅ AgentMetricsCollector ready — tracking 4 Golden Signals")

---
## 🔷 Phần 3 — Observable LoanBot v3.0

Wrap từng agent call với span + metrics recording.

In [ ]:
# ─── ObservableAgent Base — wraps every agent call

class ObservableAgent:
    """Base class: instruments every LLM call with OTel span + Prometheus metrics."""

    def __init__(self, name: str, capability: str,
                 mc: AgentMetricsCollector, tc: TraceCollector):
        self.name = name
        self.capability = capability
        self.mc = mc
        self.tc = tc

    def _call_llm(self, system: str, user: str) -> tuple[str, int, int, float]:
        """Returns (text, input_tokens, output_tokens, cost_usd)."""
        resp = client.messages.create(
            model=MODEL, max_tokens=350,
            messages=[{"role": "user", "content": user}],
            system=system
        )
        text = resp.content[0].text
        tok_in  = resp.usage.input_tokens
        tok_out = resp.usage.output_tokens
        # claude-haiku-4-5-20251001 pricing: $0.25/1M in, $1.25/1M out
        cost = (tok_in * 0.00000025) + (tok_out * 0.00000125)
        return text, tok_in, tok_out, cost

    def _parse_json(self, text: str) -> Dict:
        try:
            s, e = text.find('{'), text.rfind('}') + 1
            if s >= 0 and e > s:
                return json.loads(text[s:e])
        except: pass
        return {}

    async def run(self, payload: Dict, trace_id: str, parent_span_id: str) -> Dict:
        """Run agent with full observability instrumentation."""
        span = self.tc.start_span(
            name=self.name,
            trace_id=trace_id,
            parent_id=parent_span_id,
            **{"agent.capability": self.capability, "llm.model": MODEL}
        )
        t0 = time.time()
        error = False
        result = {}
        tok_in = tok_out = 0
        cost = 0.0
        try:
            result, tok_in, tok_out, cost = await self._process(payload)
            span.end(status="OK",
                     **{"llm.tokens.input": tok_in, "llm.tokens.output": tok_out,
                        "llm.tokens.total": tok_in + tok_out, "llm.cost.usd": round(cost, 6)})
        except Exception as ex:
            error = True
            span.end(status="ERROR", **{"error.message": str(ex)})
            result = self._fallback()
        finally:
            latency_ms = (time.time() - t0) * 1000
            self.mc.record(self.name, latency_ms, tok_in, tok_out, cost, error)
        return result

    async def _process(self, payload) -> tuple[Dict, int, int, float]:
        raise NotImplementedError

    def _fallback(self) -> Dict:
        return {"error": True, "agent": self.name}


# ─── 7 Observable Specialized Agents

class ObsCreditAgent(ObservableAgent):
    def __init__(self, mc, tc): super().__init__("CreditAgent", "credit_check", mc, tc)
    def _fallback(self): return {"credit_score": 550, "risk_grade": "C", "fallback": True}
    async def _process(self, p):
        prompt = f"Customer {p.get('customer_id')}, loan {p.get('loan_amount',0):,.0f} VND. History: {p.get('credit_history','Good')}. Return JSON: {{\"credit_score\": int, \"risk_grade\": str}}"
        text, ti, to, cost = self._call_llm("Credit scoring specialist. JSON only.", prompt)
        r = self._parse_json(text) or {"credit_score": 700, "risk_grade": "B"}
        return r, ti, to, cost

class ObsIncomeAgent(ObservableAgent):
    def __init__(self, mc, tc): super().__init__("IncomeAgent", "income_verify", mc, tc)
    def _fallback(self): return {"monthly_income": 10000000, "income_verified": False, "fallback": True}
    async def _process(self, p):
        prompt = f"Customer {p.get('customer_id')}, declared income {p.get('declared_income',0):,.0f} VND/month. Employment: {p.get('employment','Unknown')}. Return JSON: {{\"monthly_income\": float, \"income_verified\": bool, \"stability\": str}}"
        text, ti, to, cost = self._call_llm("Income verification specialist. JSON only.", prompt)
        r = self._parse_json(text) or {"monthly_income": 20000000, "income_verified": True, "stability": "HIGH"}
        return r, ti, to, cost

class ObsFraudAgent(ObservableAgent):
    def __init__(self, mc, tc): super().__init__("FraudAgent", "fraud_detection", mc, tc)
    def _fallback(self): return {"fraud_score": 0.5, "flags": ["fallback"], "recommendation": "FLAG"}
    async def _process(self, p):
        prompt = f"Customer {p.get('customer_id')}, amount {p.get('loan_amount',0):,.0f} VND. Patterns: {p.get('patterns','Normal')}. Return JSON: {{\"fraud_score\": float, \"flags\": list, \"recommendation\": str}}"
        text, ti, to, cost = self._call_llm("Fraud detection specialist. JSON only.", prompt)
        r = self._parse_json(text) or {"fraud_score": 0.05, "flags": [], "recommendation": "PROCEED"}
        return r, ti, to, cost

class ObsRiskAgent(ObservableAgent):
    def __init__(self, mc, tc): super().__init__("RiskAgent", "risk_assessment", mc, tc)
    def _fallback(self): return {"risk_score": 80.0, "decision": "REJECTED", "interest_rate": 0, "fallback": True}
    async def _process(self, p):
        prompt = f"Credit: {p.get('credit_score',600)}, Income: {p.get('monthly_income',0):,.0f}, Loan: {p.get('loan_amount',0):,.0f}, Fraud: {p.get('fraud_score',0.1)}. Return JSON: {{\"risk_score\": float, \"decision\": str, \"interest_rate\": float}}"
        text, ti, to, cost = self._call_llm("Risk assessment specialist. JSON only.", prompt)
        r = self._parse_json(text) or {"risk_score": 35.0, "decision": "APPROVED", "interest_rate": 8.5}
        return r, ti, to, cost

class ObsComplianceAgent(ObservableAgent):
    HITL_THRESHOLD = 500_000_000
    def __init__(self, mc, tc): super().__init__("ComplianceAgent", "compliance_check", mc, tc)
    def _fallback(self): return {"compliant": False, "hitl_required": True, "violations": ["fallback"]}
    async def _process(self, p):
        hitl = p.get('loan_amount', 0) > self.HITL_THRESHOLD
        violations = []
        if hitl and not p.get('human_reviewed'):
            violations.append("Art.14: HITL required for loan >500M VND")
        r = {"compliant": len(violations) == 0, "hitl_required": hitl, "violations": violations}
        return r, 0, 0, 0.0   # No LLM call — rule-based

class ObsReportAgent(ObservableAgent):
    def __init__(self, mc, tc): super().__init__("ReportAgent", "report_generation", mc, tc)
    def _fallback(self): return {"summary": "Report generation failed", "report_url": "/reports/error.pdf"}
    async def _process(self, p):
        prompt = f"Viết tóm tắt quyết định vay (3 câu ngắn, tiếng Việt). Hồ sơ: {p.get('loan_id')}, Quyết định: {p.get('decision','N/A')}, Score: {p.get('credit_score',0)}, Lãi suất: {p.get('interest_rate',0)}%/năm."
        text, ti, to, cost = self._call_llm("Chuyên gia viết báo cáo tín dụng. Ngắn gọn, chuyên nghiệp.", prompt)
        r = {"summary": text.strip(), "report_url": f"/reports/{p.get('loan_id','x')}.pdf"}
        return r, ti, to, cost

class ObsAuditAgent(ObservableAgent):
    def __init__(self, mc, tc): super().__init__("AuditAgent", "audit_logging", mc, tc)
    def _fallback(self): return {"audit_id": "AUDIT-ERROR", "stored": False}
    async def _process(self, p):
        audit_id = f"AUDIT-{datetime.now().strftime('%Y%m%d%H%M%S')}"
        r = {"audit_id": audit_id, "stored": True, "retention": "3 years (EU AI Act Art.12)"}
        return r, 0, 0, 0.0   # No LLM call


print("✅ 7 Observable Agents defined — đầy đủ span + metrics instrumentation")

In [ ]:
# ─── ObservableSupervisor — orchestrates với full tracing

class ObservableSupervisor:
    def __init__(self, mc: AgentMetricsCollector, tc: TraceCollector):
        self.mc = mc
        self.tc = tc
        self.agents = {
            "credit":     ObsCreditAgent(mc, tc),
            "income":     ObsIncomeAgent(mc, tc),
            "fraud":      ObsFraudAgent(mc, tc),
            "risk":       ObsRiskAgent(mc, tc),
            "compliance": ObsComplianceAgent(mc, tc),
            "report":     ObsReportAgent(mc, tc),
            "audit":      ObsAuditAgent(mc, tc),
        }

    async def process(self, app: Dict) -> Dict:
        loan_id = app['loan_id']
        trace_id = str(uuid.uuid4())[:12]

        # Root span: Supervisor
        root_span = self.tc.start_span("IntelligentSupervisor", trace_id,
                                        **{"loan.id": loan_id, "llm.model": "n/a"})

        print(f"\n🏢 Processing {loan_id} | trace: {trace_id}")

        # Phase 1: parallel
        print("  ⚡ Phase 1 (parallel): Credit + Income + Fraud")
        p1 = await asyncio.gather(
            self.agents["credit"].run(app, trace_id, root_span.span_id),
            self.agents["income"].run(app, trace_id, root_span.span_id),
            self.agents["fraud"].run(app, trace_id, root_span.span_id),
        )
        results = {}
        for r in p1: results.update(r)

        # Phase 2: parallel
        print("  ⚡ Phase 2 (parallel): Risk + Compliance")
        phase2_payload = {**app, **results}
        p2 = await asyncio.gather(
            self.agents["risk"].run(phase2_payload, trace_id, root_span.span_id),
            self.agents["compliance"].run(phase2_payload, trace_id, root_span.span_id),
        )
        for r in p2: results.update(r)

        # HITL gate
        if results.get('hitl_required') and not app.get('human_reviewed'):
            root_span.end(status="OK", **{"hitl.gate": True})
            return {"status": "PENDING_HUMAN_REVIEW", "loan_id": loan_id, "trace_id": trace_id}

        # Phase 3: parallel
        print("  ⚡ Phase 3 (parallel): Report + Audit")
        phase3_payload = {**app, **results}
        p3 = await asyncio.gather(
            self.agents["report"].run(phase3_payload, trace_id, root_span.span_id),
            self.agents["audit"].run(phase3_payload, trace_id, root_span.span_id),
        )
        for r in p3: results.update(r)

        root_span.end(status="OK", **{"pipeline.complete": True})
        self.mc.record_app_complete()

        return {
            "status": "COMPLETED", "loan_id": loan_id, "trace_id": trace_id,
            "decision": results.get('decision', 'N/A'),
            "credit_score": results.get('credit_score', 0),
            "interest_rate": results.get('interest_rate', 0),
            "audit_id": results.get('audit_id', ''),
            "summary": results.get('summary', ''),
        }


print("✅ ObservableSupervisor defined")

---
## 🔷 Phần 4 — Chạy 3 Hồ Sơ & Xem Dashboard

In [ ]:
# ─── Chạy 3 hồ sơ qua Observable LoanBot v3.0
sup = ObservableSupervisor(metrics, tracer)

apps = [
    {
        "loan_id": "FC-2024-001", "customer_id": "CUST-001",
        "loan_amount": 150_000_000, "declared_income": 25_000_000,
        "credit_history": "5 years no default, excellent",
        "employment": "Senior engineer, 7 years",
        "patterns": "Regular, consistent behavior"
    },
    {
        "loan_id": "FC-2024-002", "customer_id": "CUST-002",
        "loan_amount": 80_000_000, "declared_income": 18_000_000,
        "credit_history": "1 late payment 2 years ago",
        "employment": "Government employee, 3 years",
        "patterns": "First-time loan applicant"
    },
    {
        "loan_id": "FC-2024-003", "customer_id": "CUST-003",
        "loan_amount": 300_000_000, "declared_income": 45_000_000,
        "credit_history": "Good, 4 years",
        "employment": "Business owner, 10 years",
        "patterns": "Multiple applications in 3 months"
    }
]

results = []
for app in apps:
    r = await sup.process(app)
    results.append(r)
    print(f"  → {r['loan_id']}: {r.get('decision', r.get('status'))} | Score: {r.get('credit_score', 'N/A')}")

print("\n✅ Đã xử lý 3 hồ sơ!")

In [ ]:
# ─── Xem Dashboard
metrics.print_dashboard()

In [ ]:
# ─── Xem Trace Waterfall của hồ sơ đầu tiên
first_trace_id = results[0]['trace_id']
tracer.print_waterfall(first_trace_id)

---
## 🔷 Phần 5 — SLO Monitor & Alerting

In [ ]:
# ─── SLO Monitor — check và alert theo ngưỡng

class SLOMonitor:
    """Check SLOs sau mỗi batch xử lý. Production → alert to PagerDuty/Slack."""

    def __init__(self, mc: AgentMetricsCollector):
        self.mc = mc
        self.alerts_fired: List[Dict] = []

    def _alert(self, severity: str, name: str, message: str, action: str):
        alert = {"severity": severity, "name": name, "message": message,
                 "action": action, "timestamp": datetime.now().isoformat()}
        self.alerts_fired.append(alert)
        icon = {"CRITICAL": "🔴", "WARNING": "🟡", "INFO": "🔵"}.get(severity, "⚪")
        print(f"  {icon} [{severity}] {name}: {message}")
        print(f"     ↳ Action: {action}")

    def evaluate(self):
        print("\n🚨 SLO Monitor — Evaluating Alerts:")
        alerts_before = len(self.alerts_fired)

        for agent_name, slo_ms in self.mc.SLO_LATENCY_MS.items():
            p99 = self.mc.p(agent_name, 99)
            if p99 == 0: continue

            if p99 > slo_ms:
                self._alert("CRITICAL", f"{agent_name} SLO Breach",
                            f"P99={p99:.0f}ms > SLO={slo_ms}ms",
                            "Page on-call, check circuit breaker, scale up")
            elif p99 > slo_ms * 0.9:
                self._alert("WARNING", f"{agent_name} Approaching SLO",
                            f"P99={p99:.0f}ms at {p99/slo_ms*100:.0f}% of SLO",
                            "Monitor closely, prepare runbook")

        # Error rate
        total_calls = sum(self.mc.calls.values())
        total_errors = sum(self.mc.errors.values())
        if total_calls > 0:
            er = total_errors / total_calls * 100
            if er > self.mc.SLO_ERROR_RATE:
                self._alert("CRITICAL", "Error Rate SLO Breach",
                            f"Error rate {er:.3f}% > SLO {self.mc.SLO_ERROR_RATE}%",
                            "Freeze deployments, investigate root cause")

        # Cost
        cpa = self.mc.cost_per_app()
        if cpa > self.mc.SLO_COST_PER_APP:
            self._alert("WARNING", "Cost SLO Approaching",
                        f"${cpa:.5f}/hồ sơ > SLO ${self.mc.SLO_COST_PER_APP}",
                        "Review prompt length, check model version, optimize")

        new_alerts = len(self.alerts_fired) - alerts_before
        if new_alerts == 0:
            print("  ✅ All SLOs within bounds — no alerts")
        print(f"  Total alerts fired this session: {len(self.alerts_fired)}")


slo_monitor = SLOMonitor(metrics)
slo_monitor.evaluate()

---
## 🏋️ Bài Tập Thực Hành

### Bài 1 — Simulate SLO Breach
Tạo thêm 5 hồ sơ với các conditions làm CreditAgent chậm hơn. Verify SLOMonitor fire alert đúng.

### Bài 2 — Cost Analysis
Chạy 10 hồ sơ và analyze cost breakdown: agent nào tốn nhiều nhất? Agent nào không cần LLM call?

### Bài 3 — Custom Span Attribute
Thêm attribute `loan.amount_tier` vào mỗi span: 'SMALL' (<100M), 'MEDIUM' (100M-500M), 'LARGE' (>500M). Analyze latency theo tier.

In [ ]:
# Bài 1: Simulate SLO Breach — chạy nhiều hồ sơ để accumulate latency data
# TODO: tạo batch 5 hồ sơ và chạy

# batch_apps = [
#     {"loan_id": f"FC-BATCH-{i:03d}", "customer_id": f"CUST-{i:03d}",
#      "loan_amount": (i * 50_000_000), "declared_income": 20_000_000, ...}
#     for i in range(1, 6)
# ]
# for app in batch_apps:
#     await sup.process(app)
# metrics.print_dashboard()
# slo_monitor.evaluate()

print("💡 Implement batch processing và verify SLO alerts!")

In [ ]:
# Bài 2: Cost Analysis
print("💰 Cost Analysis — After 3 Applications:")
print(f"\nTotal cost: ${metrics.total_cost:.6f}")
print(f"Apps processed: {metrics.total_apps}")
print(f"Cost per app: ${metrics.cost_per_app():.6f}\n")

print("Cost by agent (sorted):")
agent_costs = sorted(metrics.costs.items(), key=lambda x: x[1], reverse=True)
for name, cost in agent_costs:
    pct = cost / metrics.total_cost * 100 if metrics.total_cost > 0 else 0
    tokens = metrics.tokens_in[name] + metrics.tokens_out[name]
    print(f"  {name:<22} ${cost:.6f} ({pct:4.1f}%) | tokens: {tokens}")

print("\n💡 Nhận xét: ComplianceAgent và AuditAgent = $0 vì rule-based, không cần LLM")

---
## 📊 Tổng Kết Module 8

| Component | Implemented | Production equivalent |
|-----------|------------|----------------------|
| Span & Trace Collector | ✅ | OpenTelemetry SDK + Jaeger/Phoenix |
| 4 Golden Signals | ✅ | Prometheus metrics |
| Per-agent P99 latency | ✅ | Grafana histogram panel |
| Token + cost tracking | ✅ | Real Claude API usage data |
| SLO definitions | ✅ | Per-agent SLA agreements |
| Error budget | ✅ | Prometheus alert rules |
| SLO Monitor + alerting | ✅ | PagerDuty / Slack integration |
| Trace waterfall | ✅ | Jaeger / Grafana Tempo |

### 🎯 Key Numbers
- MTTD với full observability: **2.3 phút** (vs 47 phút không có)
- MTTR với trace: **23 phút** (vs 3 giờ)
- LoanBot cost: **~$0.002/hồ sơ** (~50 VNĐ) vs 350,000 VNĐ manual
- ROI observability stack: **~8M VNĐ/tháng** infrastructure → saves **15M VNĐ/phút downtime**

---
**AI Agent Harness Engineering · Module 8/8 hoàn thành**